In [1]:
# Colab only #

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install necessary packages not available by default in Colab
!pip install rouge-score peft

# In case of issues with torchao
!pip install -U --force-reinstall torchao>=0.16.0

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=fca2b347719080f7830e842b15f99018efe33679c2ae3efce070cbdf2b648611
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [2]:
# Import libraries
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, GenerationConfig, set_seed
import torch
from peft import LoraConfig, get_peft_model, TaskType
from rouge_score import rouge_scorer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Path for Colab
google_drive_data = 'drive/MyDrive/data/cleaned_data.csv'

# Local path
local_data = '../data/cleaned_data.csv'

# Set seed for reproducibility
SEED = 60
set_seed(SEED)

# Load data to DF
df = pd.read_csv(google_drive_data)
# df = pd.read_csv(local_data)
df.head()

,id,findings,impression
0,1,The cardiac silhouette and mediastinum size ar...,Normal chest [REDACTED].
1,2,Borderline cardiomegaly. Midline sternotomy [R...,No acute pulmonary findings.
2,4,There are diffuse bilateral interstitial and a...,Bullous emphysema and interstitial fibrosis. P...
3,7,The cardiac contours are normal. [REDACTED] ba...,Basilar atelectasis. No confluent lobar consol...
4,9,The [REDACTED] examination consists of frontal...,Increased size of density in the left cardioph...


In [4]:
# Regex patterns to classify normal and abnormal cases
normal_case_regex = r'no acute|no evidence of|negative|clear lungs|normal|unremarkable'
abnormal_case_regex = r'atelectasis|effusion|cardiomegaly|congestion|pneumothorax'

# Categorize cases based on regex
case_filter = df['impression'].str.contains(normal_case_regex, case=False, regex=True) & ~df['impression'].str.contains(abnormal_case_regex, case=False, regex=True)
df['key'] = np.where(case_filter, 'normal', 'abnormal')

df.head()

,id,findings,impression,key
0,1,The cardiac silhouette and mediastinum size ar...,Normal chest [REDACTED].,normal
1,2,Borderline cardiomegaly. Midline sternotomy [R...,No acute pulmonary findings.,normal
2,4,There are diffuse bilateral interstitial and a...,Bullous emphysema and interstitial fibrosis. P...,abnormal
3,7,The cardiac contours are normal. [REDACTED] ba...,Basilar atelectasis. No confluent lobar consol...,abnormal
4,9,The [REDACTED] examination consists of frontal...,Increased size of density in the left cardioph...,abnormal


In [5]:
# Train Test Validation Split 80/10/10
train_df, val_test_df = train_test_split(df, test_size=0.2, stratify=df['key'], random_state=SEED)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['key'], random_state=SEED)

# Print the number of normal and abnormal cases in the training set
print(f'Normal cases: {len(train_df[train_df['key']=='normal'])}')
print(f'Abnormal cases: {len(train_df[train_df['key']=='abnormal'])}')

Normal cases: 922
Abnormal cases: 765


In [6]:
# Load base model & tokenizer
model_name = 'google-t5/t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Add special token for [REDACTED]
tokenizer.add_special_tokens({'additional_special_tokens': ['[REDACTED]']})
model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Embedding(32101, 768)

In [7]:
# Tokenize data function
def tokenize_data(data):
    model_inputs = tokenizer(data['findings'], max_length=512,truncation=True)
    labels = tokenizer(data['impression'], max_length=512, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Convert DF to dataset
train_dataset = Dataset.from_pandas(train_df.drop(columns=['key']), preserve_index=False)
test_dataset = Dataset.from_pandas(test_df.drop(columns=['key']), preserve_index=False)
val_dataset = Dataset.from_pandas(val_df.drop(columns=['key']), preserve_index=False)

# Tokenize dataset
train_dataset_tokenized = train_dataset.map(tokenize_data, batched=True, remove_columns=train_dataset.column_names)
test_dataset_tokenized = test_dataset.map(tokenize_data, batched=True, remove_columns=test_dataset.column_names)
val_dataset_tokenized = val_dataset.map(tokenize_data, batched=True, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/1687 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

In [8]:
# Define lora config
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=['q','k','v','o'],
    lora_dropout=0.5,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Apply lora config to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 14,155,776 || all params: 237,038,592 || trainable%: 5.9719


In [9]:
# Define generation config
gen_conf = GenerationConfig(
    max_new_tokens=128,
    num_beams=4,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    early_stopping=True
)

In [10]:
# Path for Colab
google_path_model = '/content/drive/MyDrive/models/t5'

# Local path
local_path_model = '../models/t5'

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=google_path_model,
    # output_dir=local_path_model,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=15,
    logging_steps=10,
    seed=SEED,
    load_best_model_at_end=True
)

# Define data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [11]:
# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

In [12]:
# Start training    
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.171048,2.948784
2,3.208164,2.676587
3,2.517723,2.529258
4,2.674590,2.410434
5,2.544801,2.339101
6,2.494244,2.277159
7,2.305087,2.231532
8,2.204641,2.201200
9,2.201103,2.166544
10,2.512687,2.145362


/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

TrainOutput(global_step=1590, training_loss=2.5160338683698162, metrics={'train_runtime': 476.3948, 'train_samples_per_second': 53.118, 'train_steps_per_second': 3.338, 'total_flos': 3964110935694336.0, 'train_loss': 2.5160338683698162, 'epoch': 15.0})

In [13]:
trainer.evaluate(eval_dataset=test_dataset_tokenized)

Training Loss,Validation Loss,Epoch
2.213913,2.192998,15


{'eval_loss': 2.1929984092712402}

In [14]:
# Test rougr1_f1 score
def r1_f1(data):
    total=0
    count=0
    for i in data:
        prediction = model.generate(**tokenizer(i['findings'], return_tensors='pt', truncation=True, max_length=512, generation_config=gen_conf).to(model.device), max_length=512)
        prediction_text = tokenizer.decode(prediction[0], skip_special_tokens=True)
        label_text = i['impression']

        score = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True).score(label_text, prediction_text)

        total+= score['rouge1'].fmeasure
        count+= 1
    return total / count if count > 0 else 0

In [15]:
# Score rouge1 f1 on validation and test dataset 
val_r1_f1 = r1_f1(val_dataset)
test_r1_f1 = r1_f1(test_dataset)

print(f'Validation dataset R1 F1: {val_r1_f1}')
print(f'Test dataset R1 F1: {test_r1_f1}')

Validation dataset R1 F1: 0.3275711337051968
Test dataset R1 F1: 0.38838101500978667


In [16]:
# Test model on given examples
test1 = 'The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities.'
test2 = 'The lungs are clear. Heart size and mediastinal contours are normal. No osseous abnormalities.'
test3 = 'AP and lateral views were obtained. Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly. No pneumothorax. Mild pulmonary vascular congestion'

for test in [test1, test2, test3]:
    prediction = model.generate(**tokenizer(test, return_tensors='pt', truncation=True, max_length=512, generation_config=gen_conf).to(model.device), max_length=512)
    prediction_text = tokenizer.decode(prediction[0], skip_special_tokens=True)
    print(f'Input: {test}')
    print(f'Output: {prediction_text}\n')

Input: The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities.
Output: No acute cardiopulmonary abnormalities.

Input: The lungs are clear. Heart size and mediastinal contours are normal. No osseous abnormalities.
Output: No acute cardiopulmonary findings.

Input: AP and lateral views were obtained. Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly. No pneumothorax. Mild pulmonary vascular congestion
Output: Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly.



In [17]:
# Colab save path
google_ftmodel_path = 'drive/MyDrive/models/t5-summarization'
# Local save path
local_ftmodel_path = '../model/t5-summarization'

# Save fine tuned model
updated_model = model.merge_and_unload()
updated_model.save_pretrained(google_ftmodel_path)
tokenizer.save_pretrained(google_ftmodel_path)
# updated_model.save_pretrained(local_ftmodel_path)
# tokenizer.save_pretrained(local_ftmodel_path)

updated_model.eval()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32101, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32101, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro